# Robustness of Accessibility
## Notebook 4/4 - Anlyse and visualize scores

In [ ]:
from __future__ import annotations

import branca as branca
import matplotlib.pyplot as plt
import pandas as pd
import folium
import geopandas as gpd

In [ ]:
from pathlib import Path
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    get_roa_cache_path,
    get_roa_data_path,
    get_roa_outputs_path,
    RoaNotebookConfig,
)

output_dir: Path = get_roa_outputs_path()
cache_dir: Path = get_roa_cache_path()
place_name: str = RoaNotebookConfig.place_name
data_path: Path = get_roa_data_path()

In [ ]:
# output dir is
print(output_dir)

In [ ]:
# Copy cell output from 03_RobustnessOfAccessibility.ipynb to get reference of the run to investigate or copy from local file name

now = "2025-11-10 09:36:32.951420"
scores_fire_stations = pd.read_csv(output_dir / f"scores_fire_stations_{now}.csv")
scores_hospitals = pd.read_csv(output_dir / f"scores_hospitals_{now}.csv")

# Load the administrative boundaries in order to assign each node to the relevant districts for later analysis

In [ ]:
administrative_districts = gpd.read_file(cache_dir / f"services/administrative_districts_{place_name}.geojson")
administrative_districts.plot()

In [ ]:
administrative_districts.reset_index(inplace=True)
# only use boundaries that have a name
gdf_boundaries_with_name = administrative_districts[administrative_districts["name"].notnull()]

In [ ]:
from shapely import wkt


def parse_scores_to_gdf(scores):
    scores["geometry"] = scores["to_centroid"]
    scores["geometry"] = scores["geometry"].apply(wkt.loads)
    scores = scores.set_geometry("geometry")
    scores = scores.set_crs("EPSG:4326")
    return scores

In [ ]:
scores_fire_stations = parse_scores_to_gdf(scores_fire_stations)
scores_hospitals = parse_scores_to_gdf(scores_hospitals)

In [ ]:
def add_district_names_to_scores(admin_boundaries: gpd.GeoDataFrame, scores: pd.DataFrame):
    # gdf_points: Punkte-GeodataFrame
    # gdf_polygons: Polygon-GeodataFrame mit Spalte "name"

    # sicherstellen, dass beide im selben CRS sind
    scores = scores.to_crs(admin_boundaries.crs)

    # räumliche Verknüpfung: welcher Punkt liegt in welchem Polygon
    gdf_joined = gpd.sjoin(scores, admin_boundaries[["name", "geometry"]], predicate="within", how="left")

    # Ergebnis enthält für jeden Punkt die Spalte 'name' des Distrikts
    return gdf_joined

In [ ]:
scores_fire_stations: pd.DataFrame = add_district_names_to_scores(
    admin_boundaries=gdf_boundaries_with_name, scores=scores_fire_stations
)
scores_hospitals: pd.DataFrame = add_district_names_to_scores(
    admin_boundaries=gdf_boundaries_with_name, scores=scores_hospitals
)
scores_fire_stations.head()

# Results
Summary of the results:

## Scores by districts

In [ ]:
# TODO goup scores by districts i.e. goup dataframe using name column
scores_hospitals_by_district = scores_hospitals.groupby("name")["score"].mean().reset_index()
scores_fire_stations_by_district = scores_fire_stations.groupby("name")["score"].mean().reset_index()

In [ ]:
scores_hospitals_by_district

In [ ]:
scores_fire_stations_by_district

In [ ]:
#  overall score RoA
average_score_fire_stations = scores_fire_stations["score"].mean()
average_score_hospitals = scores_hospitals["score"].mean()
average_score_total = (average_score_hospitals + average_score_fire_stations) / 2


round_numbers = 3
overall_results = {
    "score_fire_stations": round(average_score_fire_stations, round_numbers),
    "score_hospitals": round(average_score_hospitals, round_numbers),
    "score_total": round(average_score_total, round_numbers),
    # "% disrupted fire stations": round(disrupted_relative_fire, round_numbers),
    # "% disrupted hospitals": round(disrupted_relative_hospitals, round_numbers),
    # "additional_distance_fire_stations_connected": round(additional_distance_fire_stations_connected, round_numbers),
    # "additional_distance_hospitals_connected": round(additional_distance_hospitals_connected, round_numbers),
}
overall_results

## Visualization

In [ ]:
def get_color(score_value: float):
    g = min(255.0, score_value * 510)
    r = min(255.0, (1 - score_value) * 510)
    b = 0
    return f"rgb({r},{g},{b})"

In [ ]:
icons = [
    "icons/circle_filled.png",
    "icons/plus.png",
    "icons/triangle_filled_grey.png",
    "icons/square_filled_grey.png",
    "icons/circle.png",
]
labels = ["RoA = 0.0", "0 < RoA <= 0.25", "0.25 < RoA <= 0.75", "0.75 < RoA < 1", "RoA = 1.0"]


def get_icon_path(score_value: float):
    if score_value == 0.0:
        path = icons[0]
    elif score_value <= 0.25:
        path = icons[1]
    elif score_value < 0.75:
        path = icons[2]
    elif score_value < 1:
        path = icons[3]
    else:
        path = icons[4]
    return path

In [ ]:
def add_categorical_legend(folium_map, title, icons, labels, icon_size=10, text_size=10, margin_top=0):
    # copied from https://stackoverflow.com/questions/65042654/how-to-add-categorical-legend-to-python-folium-map
    if len(icons) != len(labels):
        raise ValueError("icons and labels must have the same length.")

    icons_by_label = dict(zip(labels, icons))

    legend_categories = ""
    for label, icon in icons_by_label.items():
        legend_categories += (
            f"<li><img src='{icon}' style='width: {icon_size}px; height: {icon_size}px;'></img> {label}</li>"
        )

    legend_html = f"""
    <div id='maplegend' class='maplegend'>
      <div class='legend-title'>{title}</div>
      <div class='legend-scale'>
        <ul class='legend-labels'>
        {legend_categories}
        </ul>
      </div>
    </div>
    """
    script = f"""
        <script type="text/javascript">
        var oneTimeExecution = (function() {{
                    var executed = false;
                    return function() {{
                        if (!executed) {{
                             var checkExist = setInterval(function() {{
                                       if ((document.getElementsByClassName('leaflet-bottom leaflet-left').length) || (!executed)) {{
                                          document.getElementsByClassName('leaflet-bottom leaflet-left')[0].style.display = "flex"
                                          document.getElementsByClassName('leaflet-bottom leaflet-left')[0].style.flexDirection = "column"
                                          document.getElementsByClassName('leaflet-bottom leaflet-left')[0].innerHTML += `{legend_html}`;
                                          clearInterval(checkExist);
                                          executed = true;
                                       }}
                                    }}, 100);
                        }}
                    }};
                }})();
        oneTimeExecution()
        </script>
      """
    # padding ist der Bereich oben und unten von der Karte
    css = f"""

    <style type='text/css'>
      .maplegend {{
        z-index:9999;
        float:right;
        background-color: rgba(255, 255, 255, 1);
        border-radius: 5px;
        border: 2px solid #bbb;
        margin-top: {margin_top}px;
        padding: 10px;
        font-size:{text_size}px;
        positon: relative;
      }}
      .maplegend .legend-title {{
        text-align: left;
        margin-bottom: 5px;
        font-weight: bold;
        font-size: 90%;
        }}
      .maplegend .legend-scale ul {{
        margin: 0;
        margin-bottom: 5px;
        padding: 0;
        float: left;
        list-style: none;
        }}
      .maplegend .legend-scale ul li {{
        font-size: 80%;
        list-style: none;
        margin-left: 0;
        line-height: {text_size + 10}px;
        margin-bottom: 2px;
        }}
      .maplegend ul.legend-labels li span {{
        display: block;
        float: left;
        height: 16px;
        width: 30px;
        margin-right: 5px;
        margin-left: 0;
        border: 0px solid #ccc;
        }}
      .maplegend .legend-source {{
        font-size: 80%;
        color: #777;
        clear: both;
        }}
      .maplegend a {{
        color: #777;
        }}
    </style>
    """

    folium_map.get_root().header.add_child(folium.Element(script + css))

    return folium_map

In [ ]:
print_layout = False
# location_for_map = (49.7610000, 6.675) # for width = 1500
location_for_map = (49.7610000, 6.675)
print_size_h = 1600
print_size_w = 2400
# tiles = "CartoDB Positron"
# tiles = "Stamen Terrain"
tiles = "OpenStreetMap"

if print_layout:
    zoom_start = 13
    icon_size = 22
    legend_icon_size = 45
    legend_text_size = 55
else:
    zoom_start = 12
    icon_size = 12
    legend_icon_size = 25
    legend_text_size = 30

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union

In [ ]:
hospitals = gpd.read_file(cache_dir / f"services/hospitals_{place_name}.geojson")
fire_stations = gpd.read_file(cache_dir / f"services/fire_stations_{place_name}.geojson")
hospitals = hospitals[hospitals["name"] != "Klinikum Mutterhaus der Borromäerinnen Nord"]

In [ ]:
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import get_roa_hazard_data_path


event = RoaNotebookConfig.event
hazard_data_path: Path = get_roa_hazard_data_path(event=event)
# Import shapes of flooded areas
hazard_area = gpd.read_file(hazard_data_path)
hazrd_area_geoms = hazard_area.geometry

# Combine all shapes of the flooded are into one to make it easier to work with
hazard_area_multipolygon_df = gpd.GeoSeries(unary_union(hazrd_area_geoms))
hazard_area_multipolygon = hazard_area_multipolygon_df[0]

In [ ]:
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# tiles_list = ["CartoDB Positron", "Stamen Terrain", "OpenStreetMap"]


def init_default_map(attr=None):
    # create basemap:
    if print_layout:
        base_map = folium.Map(
            location=location_for_map,
            width=print_size_w,
            height=print_size_h,
            attr="icons from foo david",
            zoom_start=zoom_start,
            zoom_control=False,
            tiles=tiles,
        )  # CartoDB positron
    else:
        base_map = folium.Map(
            location=location_for_map,
            zoom_start=zoom_start,
            tiles=tiles,
            attr="icons from foo david attr",
            z_index_offset=250,
        )

    flooding_group = folium.FeatureGroup(name="Flooding Area").add_to(base_map)

    hazard_area_multipolygon_df.set_crs(epsg=4326, inplace=True)
    geo_json_flooded = folium.GeoJson(
        data=hazard_area_multipolygon_df,
        style_function=lambda x: {
            "fillColor": "#336699",
            "fillOpacity": 0.5,
            "color": "rgba(0, 0, 0, 0)",
        },
    )
    folium.Popup("Scenario 100 years").add_to(geo_json_flooded)
    geo_json_flooded.add_to(flooding_group)
    geo_json_flooded.add_to(flooding_group)

    # for tile in tiles_list:
    #    folium.TileLayer(tile).add_to(base_map)

    ## add hacked attribution
    # attribution_path = "icon_attribution.png"
    W, H = (300, 200)
    im = Image.new("RGBA", (W, H))
    draw = ImageDraw.Draw(im)
    msg = "icons: Flaticon.com"
    # w, h = draw.textsize(msg)
    fnt = ImageFont.truetype("/Library/Fonts/Arial.ttf", 14)
    draw.text((0, 0), msg, font=fnt, fill=(0, 0, 0))
    # im.crop((0, 0, 1.1 * w, 2 * h)).save(attribution_path, "PNG")

    # background_color = (255, 255, 255, 255)
    # image = Image.open(attribution_path)
    # background = Image.new(mode='RGBA', size=image.size, color=background_color)
    # background.paste(image, mask=image)
    # Save the intermediate image to a temporary file
    # background.save(attribution_path)
    # FloatImage(attribution_path, bottom=0, left=0).add_to(base_map)

    return base_map

In [ ]:
m_fire_stations = init_default_map(attr="Icons from foo")

score_group_fire_stations = folium.FeatureGroup(name="Score").add_to(m_fire_stations)
if print_layout:
    for idx, row in scores_fire_stations.iterrows():
        score = row["score"]
        location = (row.geometry.y, row.geometry.x)

        icon = folium.features.CustomIcon(get_icon_path(score), icon_size=(icon_size, icon_size))
        folium.Marker(location=location, icon=icon, popup=score, z_index_offset=500).add_to(score_group_fire_stations)

else:
    for idx, row in scores_fire_stations.iterrows():
        score = row["score"]
        # color = cmap(norm(score))
        color = get_color(score)
        location = (row.geometry.y, row.geometry.x)

        folium.CircleMarker(
            location=location,
            radius=5,
            color="black",
            fill=True,
            fill_color=color,
            z_index_offset=750,
        ).add_to(score_group_fire_stations)

        folium.CircleMarker(
            location=location,
            radius=3,
            color=color,
            fill=False,
            fill_color=color,
            fill_opacity=1,
            z_index_offset=751,
        ).add_to(score_group_fire_stations)

fire_station_group = folium.FeatureGroup(name="Fire Stations").add_to(m_fire_stations)

for idx, row in fire_stations.iterrows():
    location = (row.geometry.centroid.y, row.geometry.centroid.x)
    icon = folium.features.CustomIcon(
        str(data_path / "assets" / "icons" / "fire_stations.png"),
        icon_size=(icon_size * 2, icon_size * 2),
    )
    folium.Marker(location=location, icon=icon, popup=row["name"], z_index_offset=1000).add_to(fire_station_group)

# display the map
if print_layout:
    m_fire_stations = add_categorical_legend(
        m_fire_stations,
        "Fire Stations",
        icons=icons,
        labels=labels,
        icon_size=legend_icon_size,
        text_size=legend_text_size,
    )

else:
    # specify the min and max values of your data
    colormap = branca.colormap.LinearColormap(colors=["#00FF00", "#FF0000"], index=[0, 1], vmin=0, vmax=1)
    # colormap = colormap.to_step(index=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
    colormap.caption = "Hospital Score"
    colormap.add_to(m_fire_stations)

if not print_layout:
    folium.LayerControl().add_to(m_fire_stations)

m_fire_stations.save(output_dir / "map_fire_stations_test.html")
m_fire_stations

In [ ]:
m_hospitals = init_default_map()

# add points to the map, with different colors depending on the score

score_hospitals_group = folium.FeatureGroup(name="Score").add_to(m_hospitals)

if print_layout:
    for idx, row in scores_hospitals.iterrows():
        score = row["score"]
        location = (row.geometry.y, row.geometry.x)

        icon = folium.features.CustomIcon(get_icon_path(score), icon_size=(icon_size, icon_size))
        folium.Marker(
            location=location,
            icon=icon,
            popup=score,
        ).add_to(score_hospitals_group)

else:
    for idx, row in scores_hospitals.iterrows():
        score = row["score"]
        # color = cmap(norm(score))
        color = get_color(score)
        location = (row.geometry.y, row.geometry.x)
        folium.CircleMarker(location=location, radius=2, color=color, fill=True, fill_color=color).add_to(
            score_hospitals_group
        )

hospital_group = folium.FeatureGroup(name="Hospitals").add_to(m_hospitals)

for idx, row in hospitals.iterrows():
    location = (row.geometry.centroid.y, row.geometry.centroid.x)
    icon = folium.features.CustomIcon(
        str(data_path / "assets" / "icons" / "hospitals.png"),
        icon_size=(icon_size * 3, icon_size * 3),
    )
    folium.Marker(location=location, icon=icon, popup=row["name"], z_index_offset=1000).add_to(hospital_group)

if print_layout:
    m_hospitals = add_categorical_legend(
        m_hospitals,
        "Hospitals",
        icons=icons,
        labels=labels,
        icon_size=legend_icon_size,
        text_size=legend_text_size,
    )
else:
    # specify the min and max values of your data
    colormap = branca.colormap.LinearColormap(colors=["#00FF00", "#FF0000"], index=[0, 1], vmin=0, vmax=1)
    # colormap = colormap.to_step(index=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
    colormap.caption = "Hospital Score"
    colormap.add_to(m_hospitals)

if not print_layout:
    folium.LayerControl().add_to(m_hospitals)

m_hospitals.save(output_dir / "map_hospitals.html")
m_hospitals

In [ ]:
scores_hospitals["score"].hist(color="0.5")
plt.ylabel("Number of Nodes", fontsize=12)
plt.xlabel("RoA Index", fontsize=12)
plt.title("Hospitals", fontsize=13)
plt.ylim([0, 500])
plt.show()

In [ ]:
hist_fire = scores_fire_stations.score.hist(color="0.5")
plt.ylabel("Number of Nodes", fontsize=12)
plt.xlabel("RoA Index", fontsize=12)
plt.title("Fire Stations", fontsize=13)
plt.ylim([0, 500])
plt.show()

In [ ]:
icons_legend = [
    "icons/normal_public.png",
    "icons/normal_private.png",
    "icons/river_area_col.png",
    "icons/flooded_area_col.png",
    "icons/flooded_public_col.png",
    "icons/flooded_private_col.png",
]
labels_legend = [
    "Public Roads",
    "Private Roads",
    "Rivers",
    "Flooded Area",
    "Flooded Public Roads ",
    "Flooded Private Roads",
]

m_legend = folium.Map(location=location_for_map, zoom_start=zoom_start, zoom_control=False)  # CartoDB positron

m_legend = add_categorical_legend(
    m_legend, "", icons=icons_legend, labels=labels_legend, icon_size=legend_icon_size, text_size=legend_text_size
)

m_legend

In [ ]:
scores_hospitals.head()

In [ ]:
data = [(point.y, point.x, score) for point, score in zip(scores_hospitals["geometry"], 1 - scores_hospitals["score"])]
# data

In [ ]:
m_heat = folium.Map(location=location_for_map, zoom_start=zoom_start, zoom_control=False)  # CartoDB positron

In [ ]:
from folium.plugins import HeatMap

min_inverted_score = 1
max_inverted_score = 0

# Add the HeatMap layer to the map
HeatMap(data, min_opacity=0.3, max_val=max_inverted_score, min_val=min_inverted_score).add_to(m_heat)

m_heat